# 🧪 Cuaderno 04 — Análisis del Caso de Estudio

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Aquí resuelves el caso asignado en el **Cuaderno 03**. Para cada método numérico:

1. **Escribe tu propia versión** en un archivo `.py` (a mano, siguiendo las ecuaciones de `docs/metodos/`).
2. **Compárala** con la versión de **NumPy/SciPy**.
3. Usa el `%%writefile` de abajo como **ayuda** (léelo, entiéndelo y luego escribe el tuyo).

> 💡 Los métodos son: **Eliminación Gaussiana**, **LU**, **Gauss-Seidel** y **Thomas/Cholesky**.


In [ ]:
#@title ⚙️ Cargar entorno (ejecuta PRIMERO)
import os, sys, urllib.request
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

ARCHIVO = "robomat.py"
URL = "https://raw.githubusercontent.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/main/lib/robomat.py"


def _buscar_robomat(nombre=ARCHIVO, saltos=6):
    """Busca robomat.py junto al notebook, en la carpeta actual y en lib/ de los padres."""
    bases = [os.getcwd()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    if ruta_nb:
        bases.append(os.path.dirname(os.path.abspath(ruta_nb)))
    for base in list(bases):
        b = base
        for _ in range(saltos):
            for cand in (os.path.join(b, nombre), os.path.join(b, "lib", nombre)):
                if os.path.exists(cand):
                    return os.path.abspath(cand)
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


ruta_robomat = _buscar_robomat()
if ruta_robomat is None:
    print("robomat.py no está en disco; descargando del repositorio...")
    urllib.request.urlretrieve(URL, ARCHIVO)
    ruta_robomat = os.path.abspath(ARCHIVO)

_carpeta_lib = os.path.dirname(ruta_robomat)
if _carpeta_lib not in sys.path:
    sys.path.insert(0, _carpeta_lib)

# Import real (no exec): Pylance lo resuelve y da autocompletado.
import importlib
import robomat as _robomat
importlib.reload(_robomat)
from robomat import *               # noqa: F401,F403

print(f"✅ Entorno cargado desde: {ruta_robomat}")
import numpy as np
import scipy.linalg as sla
print("✅ Entorno cargado")


## 1. Sistema de ejemplo (reemplázalo por TU caso)

Define la **matriz de coeficientes** $\mathbf{A}$ y el **vector** $\mathbf{b}$ de tu caso.
Aquí usamos un ejemplo $3\times3$ de Chapra; en tu caso, construye $\mathbf{A}$ y $\mathbf{b}$
a partir del modelo físico de tu ficha `caso_tXX.md`.


In [ ]:
# ===== REEMPLAZA A y b por los de TU caso =====
A = np.array([[ 3.0, -0.1, -0.2],
              [ 0.1,  7.0, -0.3],
              [ 0.3, -0.2, 10.0]])
b = np.array([7.85, -19.3, 71.4])

x_ref = np.linalg.solve(A, b)
print("Solución de referencia (numpy):", np.round(x_ref, 6))
print("Cond(A) =", np.linalg.cond(A))


## 2. Eliminación Gaussiana (con pivoteo parcial)

📖 Ver [01_eliminacion_gaussiana.md](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/01_eliminacion_gaussiana.md)


In [ ]:
%%writefile gauss.py
import numpy as np

def eliminacion_gaussiana(A, b):
    '''Resuelve Ax=b por eliminacion gaussiana con pivoteo parcial.'''
    A = np.array(A, dtype=float).copy()
    b = np.array(b, dtype=float).copy().ravel()
    n = len(b)

    # Fase 1: eliminacion hacia adelante
    for k in range(n - 1):
        # --- Pivoteo parcial: mayor |a| en la columna k ---
        p = k + np.argmax(np.abs(A[k:, k]))
        if p != k:
            A[[k, p]] = A[[p, k]]
            b[[k, p]] = b[[p, k]]

        # --- Eliminar debajo del pivote ---
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]          # multiplicador m_ik
            A[i, k:] = A[i, k:] - m * A[k, k:]
            b[i] = b[i] - m * b[k]

    # Fase 2: sustitucion hacia atras
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]
    return x


In [ ]:
# Prueba tu gauss.py contra numpy
import gauss
x = gauss.eliminacion_gaussiana(A, b)
print("gauss.py        =", np.round(x, 6))
print("numpy           =", np.round(x_ref, 6))
print("error absoluto  =", np.max(np.abs(x - x_ref)))


## 3. Descomposición LU (Doolittle)

📖 Ver [02_descomposicion_lu.md](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/02_descomposicion_lu.md)


In [ ]:
%%writefile lu.py
import numpy as np

def descomposicion_lu(A):
    '''Factorizacion LU (Doolittle): A = L @ U, con diagonal de L unitaria.'''
    A = np.array(A, dtype=float).copy()
    n = A.shape[0]
    L = np.eye(n)
    U = np.zeros((n, n))

    for k in range(n):
        U[k, k:] = A[k, k:]
        for i in range(k + 1, n):
            L[i, k] = A[i, k] / U[k, k]      # multiplicador m_ik
            A[i, k:] = A[i, k:] - L[i, k] * U[k, k:]
    return L, U

def resolver_lu(A, b):
    '''Resuelve Ax=b: (1) Ly=b por sustitucion adelante, (2) Ux=y atras.'''
    L, U = descomposicion_lu(A)
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - np.dot(L[i, :i], y[:i])
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x


In [ ]:
# Prueba tu lu.py contra scipy
import lu
x = lu.resolver_lu(A, b)
L, U = lu.descomposicion_lu(A)
P, Ls, Us = sla.lu(A)   # scipy: A = P @ L @ U
print("lu.py           =", np.round(x, 6))
print("A == L@U ?", np.allclose(A, L @ U))
print("scipy P@L@U == A ?", np.allclose(A, P @ Ls @ Us))


## 4. Gauss-Seidel (iterativo, con relajación)

📖 Ver [03_metodos_iterativos.md](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/03_metodos_iterativos.md)
> ⚠️ Requiere **dominancia diagonal estricta**. Si tu matriz no la tiene, reordena los renglones.


In [ ]:
%%writefile gauss_seidel.py
import numpy as np

def gauss_seidel(A, b, x0=None, tol=1e-6, max_iter=1000, w=1.0):
    '''Resuelve Ax=b por Gauss-Seidel (relajacion opcional w en (0,2)).'''
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).ravel()
    n = len(b)
    x = np.zeros(n) if x0 is None else np.array(x0, dtype=float).ravel()

    for it in range(max_iter):
        x_old = x.copy()
        for i in range(n):
            s = np.dot(A[i, :i], x[:i]) + np.dot(A[i, i+1:], x_old[i+1:])
            x_new = (b[i] - s) / A[i, i]
            x[i] = w * x_new + (1 - w) * x_old[i]   # relajacion SOR

        # Criterio de paro: error relativo aproximado maximo (%)
        denom = np.where(x == 0, 1.0, x)
        ea = np.max(np.abs((x - x_old) / denom)) * 100
        if ea < tol:
            return x, it + 1
    return x, max_iter


In [ ]:
# Prueba tu gauss_seidel.py
import gauss_seidel
x, it = gauss_seidel.gauss_seidel(A, b, tol=1e-10)
print("gauss_seidel.py =", np.round(x, 6), "| iteraciones =", it)
print("numpy           =", np.round(x_ref, 6))


## 5. Algoritmo de Thomas (tridiagonales)

📖 Ver [04_matrices_especiales.md](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/04_matrices_especiales.md)
> 💡 Solo aplica si tu matriz es **tridiagonal**. Si no, usa Cholesky o salta este método.


In [ ]:
%%writefile thomas.py
import numpy as np

def thomas(e, f, g, r):
    '''Algoritmo de Thomas O(n) para sistemas tridiagonales.
    e: subdiagonal (e[0] sin usar), f: diagonal, g: superdiagonal, r: lado derecho.'''
    n = len(f)
    e = np.array(e, dtype=float)
    f = np.array(f, dtype=float).copy()
    g = np.array(g, dtype=float)
    r = np.array(r, dtype=float).copy()

    # Fase 1: descomposicion + sustitucion adelante
    for k in range(1, n):
        fac = e[k] / f[k - 1]
        f[k] = f[k] - fac * g[k - 1]
        r[k] = r[k] - fac * r[k - 1]

    # Fase 2: sustitucion hacia atras
    x = np.zeros(n)
    x[-1] = r[-1] / f[-1]
    for k in range(n - 2, -1, -1):
        x[k] = (r[k] - g[k] * x[k + 1]) / f[k]
    return x


In [ ]:
# Prueba thomas.py con un sistema tridiagonal de ejemplo
f = [2.0, 2.0, 2.0]
e = [0.0, -1.0, -1.0]
g = [-1.0, -1.0, 0.0]
r = [1.0, 0.0, 1.0]
import thomas
xt = thomas.thomas(e, f, g, r)
At = np.array([[2,-1,0],[-1,2,-1],[0,-1,2.0]])
print("thomas.py =", xt, "| numpy =", np.linalg.solve(At, r))


## 6. Monte Carlo (análisis de sensibilidad)

Perturba $\mathbf{b}$ con ruido y observa cómo cambia la solución. Si $\text{Cond}(\mathbf{A})$
es grande, pequeñas perturbaciones producen grandes cambios (sistema **mal condicionado**).


In [ ]:
N = 500
ruido = 0.02   # 2% de ruido relativo
X = []
for _ in range(N):
    bp = b * (1 + ruido * np.random.randn(len(b)))
    X.append(np.linalg.solve(A, bp))
X = np.array(X)

for j in range(len(b)):
    print(f"x{j+1}: media={X[:,j].mean():.4f}  desv={X[:,j].std():.4f}  "
          f"ref={x_ref[j]:.4f}")

fig, ax = plt.subplots()
ax.boxplot(X, tick_labels=[f"x{i+1}" for i in range(len(b))])
ax.set_title("Sensibilidad de la solución ante ruido en b")
ax.set_ylabel("valor")
plt.show()


## 7. Exportar resultados a LaTeX (para el reporte)


In [ ]:
def a_latex_resultados(A, b, x, metodos):
    '''Genera un snippet .tex con la matriz, el vector y la tabla de métodos.'''
    L = []
    L.append("Matriz $\\mathbf{A}$ y vector $\\mathbf{b}$ de tu caso:\\par")
    L.append("$\\mathbf{A} = " + "\\begin{bmatrix} " + " ".join(
        " & ".join(f"{v:g}" for v in fila) + " \\\\ " for fila in A) + "\\end{bmatrix}$\\par")
    L.append("$\\mathbf{b} = " + "\\begin{bmatrix} " + " \\\\ ".join(f"{v:g}" for v in b) + " \\end{bmatrix}$\\par")
    L.append("\\begin{tabular}{ll}")
    L.append("Método & Solución x \\\\ \\hline")
    for nombre, val in metodos.items():
        L.append(f"{nombre} & ${', '.join(f'{v:.4g}' for v in val)}$ \\\\")
    L.append("\\end{tabular}")
    return "\n".join(L)

metodos = {
    "Gauss": gauss.eliminacion_gaussiana(A, b),
    "LU": lu.resolver_lu(A, b),
    "Gauss-Seidel": gauss_seidel.gauss_seidel(A, b, tol=1e-10)[0],
    "numpy (ref)": x_ref,
}
tex = a_latex_resultados(A, b, x_ref, metodos)
print(tex)
with open("resultados.tex", "w", encoding="utf-8") as f:
    f.write(tex)
print("\\n✅ Guardado en resultados.tex — cópialo a tu reporte (secciones/04_resultados.tex)")


## 8. Reporte y exposición

Arma tu reporte con la plantilla de [`reporte/`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/tree/main/reporte)
(`main.tex` + `secciones/`). Integra:

- `resultados.tex` (sección de resultados).
- Figuras del Monte Carlo (`boxplot`, histogramas).
- Comparación de métodos y análisis del número de condición.

Prepara una **exposición de 5 minutos** apoyada en la teoría de `docs/metodos/`.
